In [ ]:
%pip install -q pandas numpy matplotlib seaborn openpyxl sqlalchemy "psycopg[binary]" kagglehub


In [ ]:

import kagglehub
from pathlib import Path
import shutil

# Download the public dataset files
download_dir = Path(
    kagglehub.dataset_download(
        "yashdevladdha/uber-ride-analytics-dashboard"
    )
)

print("Downloaded dataset folder:", download_dir)

# Find the original CSV file
csv_files = list(download_dir.rglob("ncr_ride_bookings.csv"))

if not csv_files:
    raise FileNotFoundError(
        "ncr_ride_bookings.csv was not found. "
        "Inspect the downloaded folder and select the correct CSV."
    )

raw_path = csv_files[0]

print("CSV found:", raw_path)


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Create project folders in Colab
PROJECT = Path("/content/uber-ride-analytics")
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
VISUALS = PROJECT / "visuals"

for folder in [RAW, PROCESSED, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)

# Keep the original CSV unchanged
destination = RAW / "ncr_ride_bookings.csv"
shutil.copy2(raw_path, destination)

# Load dataset
df = pd.read_csv(destination)

# Standardize column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())
display(df.info())


In [ ]:

print("Duplicate rows:", df.duplicated().sum())

quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (
        df.isna().mean() * 100
    ).round(2),
    "unique_values": df.nunique()
})

display(quality_report)

display(df.describe(include="all").T)


In [ ]:

clean_df = df.copy()

# Remove exact duplicate records
clean_df = clean_df.drop_duplicates().copy()

# Convert date
if "date" in clean_df.columns:
    clean_df["date"] = pd.to_datetime(
        clean_df["date"], errors="coerce", dayfirst=True
    )

# Convert time and combine with date
if "time" in clean_df.columns and "date" in clean_df.columns:
    time_text = clean_df["time"].astype("string").str.strip()

    parsed_time = pd.to_datetime(
        time_text, format="%H:%M:%S", errors="coerce"
    )

    # Try HH:MM for records without seconds
    missing_time = parsed_time.isna()
    parsed_time.loc[missing_time] = pd.to_datetime(
        time_text.loc[missing_time],
        format="%H:%M",
        errors="coerce"
    )

    # Fall back to general datetime parsing
    missing_time = parsed_time.isna()
    parsed_time.loc[missing_time] = pd.to_datetime(
        time_text.loc[missing_time], errors="coerce"
    )

    clean_df["booking_timestamp"] = (
        clean_df["date"]
        + pd.to_timedelta(
            parsed_time.dt.hour * 3600
            + parsed_time.dt.minute * 60
            + parsed_time.dt.second,
            unit="s"
        )
    )

# Convert known numeric columns when present
numeric_columns = [
    "booking_value",
    "ride_distance",
    "avg_vtat",
    "avg_ctat",
    "driver_ratings",
    "customer_rating"
]

for col in numeric_columns:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(
            clean_df[col], errors="coerce"
        )

# Standardize text values
for col in clean_df.select_dtypes(
    include=["object", "string"]
).columns:
    clean_df[col] = clean_df[col].map(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

# Derive time features
if "booking_timestamp" in clean_df.columns:
    ts = clean_df["booking_timestamp"]
    clean_df["booking_hour"] = ts.dt.hour
    clean_df["booking_day"] = ts.dt.day_name()
    clean_df["booking_month"] = ts.dt.month
    clean_df["booking_date"] = ts.dt.date

# Export cleaned data
clean_path = PROCESSED / "uber_rides_cleaned.csv"
clean_df.to_csv(clean_path, index=False)

print("Cleaned shape:", clean_df.shape)
print("Saved:", clean_path)
display(clean_df.head())
